# 🎛️ DJIA — 5-Phase Setlist Generator: User Guide

Build a full club set from your analyzed library: the classic **energy arc**
(warm-up → build → peak → breakdown → comeback) chosen from *measured* data — mood dimensions,
RMS energy, spectral brightness, BPM — instead of vibes, plus a **mix sheet for every transition**
with element-onset deck timings.

```
src/ai/setlist_generator.py
│
├── Library
│   └── load_library(db_path)                 tracks + features + mood from data/djia.db
│
├── Phase machinery (pure — no audio, no DB)
│   ├── phase_quotas(n_tracks)                split N across the 5 phases
│   ├── score_phases(tracks)                  per-track suitability for each phase
│   ├── assign_phases(tracks, quotas)         global greedy assignment
│   ├── order_setlist(assignment)             transition-optimized chain
│   └── build_setlist(tracks, n_tracks)       quotas → assignment → ordering
│
├── Transition quality
│   ├── camelot_score(a, b)                   Camelot codes ('7A') — NOT note names
│   └── transition_score(a, b, ascending)     BPM 35% / key 30% / mood 20% / energy 15%
│
└── Output
    ├── mix_points_cached(track)              element-onset mix points (audio, JSON-cached)
    ├── render_report(setlist, mix_points_fn) markdown: phase plan + mix sheets
    └── generate_setlist(...)                 end-to-end, writes results/setlist_5phase.md
```

| Function | Needs DB | Needs audio | Best for |
|---|---|---|---|
| `build_setlist` | no (any track dicts) | no | experimenting with phases/ordering |
| `render_report` | no | only via default `mix_points_fn` | custom reports |
| `generate_setlist` | yes | yes (first run; cached after) | the real thing |

**Prerequisites:** every section runs **without** an analyzed DB (a synthetic library kicks in),
but the results are only interesting with your real `data/djia.db`
(`python -m src.cli analyze --data-dir data/`). Section 7 additionally wants audio files on disk.

## Sections
1. Setup
2. Load the library
3. Phase quotas — how N tracks split across the arc
4. Phase suitability scoring
5. Build the setlist
6. Transition quality
7. Element-onset mix points (audio, cached)
8. Render the report
9. Error handling reference

## 1. Setup

Run from the **repo root**. Only stdlib + pandas here; the heavy imports (librosa) are deferred
until section 7 needs audio.

In [ ]:
import logging
import os
import sys

import pandas as pd

# Make `src` importable when running from the repo root
REPO = os.path.abspath('.')
if REPO not in sys.path:
    sys.path.insert(0, REPO)

from src.ai.setlist_generator import (
    PHASE_ORDER,
    PHASE_SHARE,
    PHASE_STRATEGY,
    build_setlist,
    camelot_score,
    generate_setlist,
    load_library,
    mix_points_cached,
    phase_quotas,
    render_report,
    score_phases,
    track_name,
    transition_score,
)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s — %(message)s",
    force=True,
)

# ---- Tunables -------------------------------------------------------------
DB_PATH  = 'data/djia.db'                 # analyzed library
N_TRACKS = 28                             # set size (5-phase minimum is 5)
OUTPUT   = 'results/setlist_5phase.md'    # where generate_setlist writes

print('Setup OK — DB:', DB_PATH, '| exists:', os.path.exists(DB_PATH))

## 2. Load the library

`load_library(db_path)` returns one flat dict per fully-analyzed track (must have BPM):
`id`, `file_name`, `file_path`, `title`, `artist`, `duration`, `bpm`, `camelot_key`, `rms_mean`,
`brightness` (spectral centroid), and the six mood dimensions
(`dark`, `hypnotic`, `euphoric`, `aggressive`, `industrial`, `minimal`).

If the DB isn't there, this cell builds a **synthetic library** spanning calm-hypnotic →
aggressive-euphoric, so the rest of the guide still runs — swap in your real DB for real results.

In [ ]:
def synthetic_library(n=40):
    """Stand-in library: track 1 is the calmest, track n the hardest."""
    tracks = []
    for i in range(n):
        x = i / (n - 1)
        tracks.append({
            'id': i + 1,
            'file_name': f'track_{i + 1:02d}.mp3', 'file_path': None,
            'title': f'Synthetic Track {i + 1}', 'artist': 'Demo',
            'duration': 360.0,
            'bpm': 122.0 + 6.0 * x,
            'camelot_key': f'{(i % 12) + 1}A',
            'rms_mean': 0.10 + 0.20 * x,
            'brightness': 1000.0 + 2000.0 * x,
            'dark': 0.5, 'hypnotic': 1.0 - x, 'euphoric': x,
            'aggressive': max(0.0, x - 0.3), 'industrial': 0.3, 'minimal': 1.0 - x,
        })
    return tracks

if os.path.exists(DB_PATH):
    tracks = load_library(DB_PATH)
    source = 'real DB'
else:
    tracks = synthetic_library()
    source = 'SYNTHETIC (no DB found)'

print(f'Library : {len(tracks)} tracks  ({source})')
pd.DataFrame(tracks)[['id', 'file_name', 'bpm', 'camelot_key', 'rms_mean',
                      'hypnotic', 'euphoric', 'aggressive']].head(8)

## 3. Phase quotas — how N tracks split across the arc

`phase_quotas(n_tracks)` turns a set size into per-phase track counts using the classic
club-set proportions — the peak is the longest phase, the breakdown is a *gesture* (1 track
below 26, 2 from 26 up — never proportional, or the floor empties):

| Phase | Share | Why |
|---|---|---|
| warm-up | 20% | earn the dancefloor — hypnotic, undemanding |
| build | 29% | accumulate tension without spending it |
| peak | 36% | the payoff — longest phase, with contrast |
| breakdown | 1–2 tracks | deliberate release; a pause, not a nap |
| comeback | 15% | fast rebuild + iconic closer |

Largest-remainder rounding guarantees the quotas sum **exactly** to `n_tracks`.
Minimum set size is 5 (one track per phase) — smaller raises `ValueError`.

In [ ]:
rows = []
for n in (5, 15, 25, N_TRACKS, 30):
    q = phase_quotas(n)
    rows.append({'set size': n, **{p: q[p] for p in PHASE_ORDER}, 'sum': sum(q.values())})
pd.DataFrame(rows).set_index('set size')

## 4. Phase suitability scoring

`score_phases(tracks)` gives every track a 0–1 suitability score **for each phase**, from
measured data only. Energy (RMS), brightness (spectral centroid) and BPM are min-max normalized
*over the candidate pool*, so the profiles are library-relative — "high energy" means high
**for your collection**. What each phase wants:

| Phase | Rewarded | Penalized |
|---|---|---|
| warm-up | hypnotic, minimal | aggressive, euphoric, energy, brightness |
| build | dark, hypnotic, industrial, **mid** energy (peak at ~0.55) | euphoric |
| peak | euphoric, aggressive, energy, brightness | — |
| breakdown | low energy, low BPM, hypnotic, minimal | aggressive |
| comeback | euphoric, dark, aggressive, energy | — |

Each phase's score column is then min-max normalized so phases compete on equal footing in the
assignment step. The weights are plain sums in `score_phases()` — if a placement feels wrong when
you actually play the set, that's the function to retune.

In [ ]:
scores = score_phases(tracks)          # {track_id: {phase: score}}
by_id = {t['id']: t for t in tracks}

for phase in PHASE_ORDER:
    top = sorted(scores, key=lambda tid: scores[tid][phase], reverse=True)[:5]
    print(f'{phase.upper():10s} top 5:')
    for tid in top:
        t = by_id[tid]
        print(f'   {scores[tid][phase]:.2f}  {track_name(t)[:52]:52s} '
              f'{t["bpm"]:6.1f} BPM  rms {t.get("rms_mean") or 0:.3f}')
    print()

## 5. Build the setlist

`build_setlist(tracks, n_tracks)` runs the whole pure pipeline:

1. **Quotas** (section 3).
2. **Assignment** — *global greedy*: all `(track, phase, score)` affinities are sorted and the
   strongest claims its slot first. This matters: assigning phase-by-phase would let the warm-up
   hoard a track the peak needs more.
3. **Ordering** — phases in canonical order; the opener is the lowest-energy warm-up track; each
   next track is the best follower of the current last by `transition_score` (section 6), with a
   small bonus for **rising energy** inside warm-up/build/comeback (the ramp phases). Ordering
   chains *across* phase boundaries too, so the last build track hands off cleanly to the first
   peak track.

Returns `{'order': [track dicts with a 'phase' key], 'quotas': {...}}`.

In [ ]:
n = min(N_TRACKS, len(tracks))         # synthetic library has 40, real DB may have fewer
setlist = build_setlist(tracks, n_tracks=n)
order = setlist['order']

print(f'Quotas  : {setlist["quotas"]}')
print(f'Ordered : {len(order)} tracks\n')

pd.DataFrame([
    {'#': i + 1, 'phase': t['phase'], 'track': track_name(t)[:48],
     'bpm': round(t['bpm'], 1), 'key': t.get('camelot_key'),
     'energy': round(t.get('rms_mean') or 0, 3)}
    for i, t in enumerate(order)
]).set_index('#')

## 6. Transition quality

Two scorers:

- **`camelot_score(a, b)`** — Camelot-wheel compatibility for codes like `'7A'` (what the
  analyzer stores). 1.0 same key, 0.9 relative major/minor or adjacent same-mode (the harmonic
  sweet spot), 0.7 two steps, 0.6 diagonal, 0.3 clash-prone, 0.5 when a key is unknown.
  ⚠️ Don't confuse it with `transition_mapper._calculate_key_compatibility`, which expects
  **note names** (`'A'`) — feeding it Camelot codes silently returns the neutral 0.5.
- **`transition_score(a, b, ascending=False)`** — the blend used for ordering:
  **BPM 35% / key 30% / mood 20% / energy 15%** (the pairing-notebook weights), plus a +0.06
  bonus when `ascending=True` and energy rises.

The cell below scores every *consecutive* pair in the ordered set — the diagnostic to check
after any retuning. Above ~0.8 is a comfortable blend.

In [ ]:
# camelot_score quick reference
for a, b in [('7A', '7A'), ('7A', '7B'), ('6A', '7A'), ('12A', '1A'), ('7A', '12A'), (None, '7A')]:
    print(f'camelot_score({a!r:6}, {b!r:6}) = {camelot_score(a, b):.2f}')

# score every consecutive transition in the ordered set
trans = pd.DataFrame([
    {'from': track_name(order[i])[:32], 'to': track_name(order[i + 1])[:32],
     'phases': f"{order[i]['phase']}→{order[i + 1]['phase']}",
     'score': round(transition_score(order[i], order[i + 1]), 3),
     'key': round(camelot_score(order[i].get('camelot_key'), order[i + 1].get('camelot_key')), 2),
     'bpm_gap': round(abs(order[i]['bpm'] - order[i + 1]['bpm']), 1)}
    for i in range(len(order) - 1)
])
print(f"\nAvg transition score : {trans['score'].mean():.3f}")
print(f"Weakest transition   : {trans['score'].min():.3f}")
trans

## 7. Element-onset mix points (audio, cached)

`mix_points_cached(track)` computes the deck timings the mix sheets are made of, by running
the phrasing engine's element-onset detector on the actual audio:

| Point | Meaning | Use on the decks |
|---|---|---|
| `mix_in` | first element entry | start the incoming deck here |
| `bass_in` | first sub/low-band entry | swap the LOWs here |
| `full_on` | all detected bands have entered | blend must be done by now |
| `mix_out` | 32 bars before the end, bar-snapped | start leaving the outgoing track |

The killer number a mix sheet derives from these:
**start the incoming deck when the outgoing reaches `mix_out − (bass_in − mix_in)`** — then the
incoming bass lands *exactly* at the outgoing track's mix-out point, no bar-counting required.

First call per track loads the full MP3 (seconds); results persist in
`results/mix_points_cache.json`, so it's one load per track **ever**. Returns `{}` when the
audio file can't be found (synthetic library, moved files) — the cell below skips gracefully.

In [ ]:
def mmss(t):
    return '—' if t is None else f'{int(t // 60)}:{int(t % 60):02d}'

a, b = order[0], order[1]   # the set's first transition
have_audio = any(
    p and os.path.exists(str(p))
    for t in (a, b) for p in (t.get('file_path'), os.path.join('data', str(t.get('file_name'))))
)

if not have_audio:
    print('No audio files on disk (synthetic library?) — skipping mix-point demo.')
else:
    pts_a, pts_b = mix_points_cached(a), mix_points_cached(b)
    print(f'OUTGOING : {track_name(a)}')
    print(f'   mix_out : {mmss(pts_a.get("mix_out"))}')
    print(f'INCOMING : {track_name(b)}  ({pts_b.get("n_onsets", 0)} element entries)')
    for k in ('mix_in', 'bass_in', 'full_on'):
        print(f'   {k:8s}: {mmss(pts_b.get(k))}')
    mo, mi, bi = pts_a.get('mix_out'), pts_b.get('mix_in'), pts_b.get('bass_in')
    if mo is not None and mi is not None:
        start_at = max(0.0, mo - ((bi - mi) if bi is not None else 0.0))
        print(f'\n→ Start the incoming deck when the outgoing reaches {mmss(start_at)};')
        print(f'  swap the LOWs at {mmss(mo)} on the outgoing clock.')

## 8. Render the report

`render_report(setlist, mix_points_fn=...)` produces the markdown: header stats → phase plan
(one table per phase, with its blend-length + EQ strategy) → one mix sheet per transition.
Special cases baked in:

- the **breakdown entry** gets a *clean break* instruction instead of a blend;
- a BPM gap beyond **±8% pitch** gets a "don't beatmatch — restart the groove" warning;
- phase-boundary transitions are tagged `[build → peak]` etc.

`mix_points_fn` is injectable: pass a stub for instant rendering (below), or leave the default
(`mix_points_cached`) for real deck timings. `generate_setlist(...)` is the one-call wrapper
(load DB → build → render → write file) — same as `python -m src.cli generate-setlist`.
Flip `RUN_FULL = True` to write the real report with audio-based mix sheets.

In [ ]:
# Fast preview: stub mix points -> instant render (no audio)
stub = lambda t: {'mix_in': 0.0, 'bass_in': 30.0, 'full_on': 90.0, 'mix_out': 300.0}
report = render_report(setlist, mix_points_fn=stub)

print('\n'.join(report.splitlines()[:34]))
print(f'\n... ({len(report.splitlines())} lines total, '
      f'{report.count("Blend length") + report.count("Clean break")} mix sheets)')

# The real thing: audio-based mix points, written to OUTPUT (needs DB + audio files)
RUN_FULL = False
if RUN_FULL and os.path.exists(DB_PATH):
    path = generate_setlist(db_path=DB_PATH, n_tracks=n, output_path=OUTPUT)
    print(f'\nFull report written -> {path}')
else:
    print(f'\nRUN_FULL is False — flip it to write {OUTPUT} with real mix sheets.')

## 9. Error handling reference

| Call | Guard | Raises |
|---|---|---|
| `phase_quotas(n < 5)` | 5-phase set needs ≥ 1 track per phase | `ValueError` |
| `build_setlist` / `assign_phases` | setlist bigger than the library | `ValueError` |
| `camelot_score('garbage', ...)` | unparseable / missing key | no raise — neutral **0.5** |
| `mix_points_cached` (missing audio/BPM) | file not found or no BPM | no raise — logs a warning, returns `{}` |
| `render_report` (empty mix points) | `{}` from `mix_points_fn` | no raise — sheet falls back to phrase-boundary advice |

Design intent: **setup mistakes raise, data gaps degrade.** A bad set size is a programming
error; a missing key or moved audio file just makes that one transition less specific.

In [ ]:
# Guards raise on setup mistakes...
try:
    phase_quotas(4)
except ValueError as e:
    print(f'phase_quotas(4)          -> ValueError: {e}')

try:
    build_setlist(tracks[:6], n_tracks=20)
except ValueError as e:
    print(f'build_setlist(6, n=20)   -> ValueError: {e}')

# ...while data gaps degrade to neutral values
print(f'camelot_score(garbage)   -> {camelot_score("garbage", "7A")}   (neutral, no raise)')
print(f'camelot_score(None)      -> {camelot_score(None, "7A")}   (neutral, no raise)')

missing = {'file_name': 'does_not_exist.mp3', 'file_path': None, 'bpm': 125.0, 'duration': 300.0}
print(f'mix_points (no audio)    -> {mix_points_cached(missing)}   (empty, logged warning)')